# Track B: BRCA1 Mutation Walk -- Caduceus Embeddings

**Purpose**: Embed the BRCA1 single-point mutation walk using Caduceus (Mamba SSM)
and cache the results for cross-model analysis.

| Model | Architecture | Params | Embedding |
|-------|-------------|--------|-----------|
| **Caduceus** (PS, 131k) | Mamba SSM, RC-equivariant | 7.7M | Hidden state mean-pool (RC-invariant) |

## Setup
1. Upload `evaluation_harness.py` to `/content/`
2. Change Runtime to **GPU**
3. Run cells in order

Results are cached to `./results/interpolation_track_b/cache/` for the analysis notebook.

In [ ]:
# Cell: Install Dependencies

print('Installing core dependencies...')
!pip install -q torch transformers matplotlib seaborn pandas scipy biopython numpy

# Build mamba-ssm from source for Caduceus
print('\nBuilding mamba-ssm from source (one-time ~10 min compile)...')
!pip install causal-conv1d mamba-ssm --no-build-isolation 2>&1 | tail -5

import torch
print(f'\ntorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Done!')

In [ ]:
# Cell: Configuration

import os
import sys
import numpy as np
import torch

sys.path.insert(0, '.')

# --- Output paths ---
OUTPUT_BASE = './results/interpolation_track_b/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR   = OUTPUT_BASE + 'cache'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# --- BRCA1 ---
BRCA1_REGION_LEN = 2000   # 2kb core region for mutation walk
BRCA1_FLANKING   = 16_384 # Total region to fetch (with flanking context)
N_EXTRA_SNPS     = 120
SEED             = 320

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Output: {OUTPUT_BASE}')

# --- Caduceus model ---
CADUCEUS_MODEL = 'kuleshov-group/caduceus-ps_seqlen-131k_d_model-256_n_layer-16'
print(f'Caduceus model: {CADUCEUS_MODEL}')

In [ ]:
# Cell: Download BRCA1 Region & Define Mutation Walk Endpoints

import urllib.request
import json as _json


def fetch_brca1_region(total_len=BRCA1_FLANKING, core_len=BRCA1_REGION_LEN):
    '''Fetch BRCA1 region from UCSC (GRCh38/hg38).

    Downloads total_len bp centered on the C61G variant position.
    The central core_len bp is the "mutation zone" where SNPs are introduced.
    '''
    center = 43_104_121  # BRCA1 C61G position (GRCh38)
    start = center - total_len // 2
    end = start + total_len
    core_start = (total_len - core_len) // 2
    core_end = core_start + core_len

    url = (f'https://api.genome.ucsc.edu/getData/sequence'
           f'?genome=hg38;chrom=chr17;start={start};end={end}')

    print(f'Fetching BRCA1 region chr17:{start:,}-{end:,} ({total_len:,} bp)...')
    try:
        req = urllib.request.Request(url)
        req.add_header('User-Agent', 'GeometricTax-Experiment/1.0')
        with urllib.request.urlopen(req, timeout=30) as response:
            data = _json.loads(response.read().decode())
            seq = data.get('dna', '').upper()
            if len(seq) >= total_len * 0.9:
                seq = seq[:total_len]
                rng = np.random.default_rng(SEED)
                seq = ''.join(
                    c if c in 'ACGT' else rng.choice(list('ACGT'))
                    for c in seq
                )
                print(f'  Downloaded {len(seq)} bp from UCSC')
                print(f'  Core region: positions [{core_start}:{core_end}] ({core_len} bp)')
                return seq, core_start, core_end
    except Exception as e:
        print(f'  UCSC download failed: {e}')

    # Fallback: synthetic
    print('  Generating synthetic BRCA1-like sequence as fallback...')
    rng = np.random.default_rng(SEED + 17)
    bases = rng.choice(list('ACGT'), size=total_len, p=[0.29, 0.21, 0.21, 0.29])
    seq = ''.join(bases)
    print(f'  Generated {len(seq)} bp synthetic BRCA1-like sequence')
    return seq, core_start, core_end


def create_pathogenic_variant(full_seq, core_start, core_end,
                              n_extra_snps=N_EXTRA_SNPS, seed=SEED):
    '''Create a pathogenic variant by mutating only the core region.'''
    rng = np.random.default_rng(seed)
    bases = list(full_seq)
    core_center = (core_start + core_end) // 2
    mutated_positions = set()

    # 1. Pathogenic mutation at core center
    alt_base = 'G' if bases[core_center] != 'G' else 'C'
    print(f'  Pathogenic mutation at position {core_center}: {bases[core_center]} -> {alt_base}')
    bases[core_center] = alt_base
    mutated_positions.add(core_center)

    # 2. Additional random SNPs within core region only
    available = [i for i in range(core_start, core_end) if i not in mutated_positions]
    snp_positions = rng.choice(available, size=min(n_extra_snps, len(available)),
                                replace=False)
    snp_positions.sort()

    for pos in snp_positions:
        original = bases[pos]
        alternatives = [b for b in 'ACGT' if b != original]
        bases[pos] = rng.choice(alternatives)
        mutated_positions.add(pos)

    mutant = ''.join(bases)
    hamming = sum(a != b for a, b in zip(full_seq, mutant))
    print(f'  Total mutations: {hamming} (all within core [{core_start}:{core_end}])')
    return mutant, sorted(mutated_positions)


# Generate endpoints
full_wt_seq, CORE_START, CORE_END = fetch_brca1_region()
full_mut_seq, mutation_positions = create_pathogenic_variant(
    full_wt_seq, CORE_START, CORE_END)

print(f'\nFull sequence length: {len(full_wt_seq)}')
print(f'Core region: [{CORE_START}:{CORE_END}] ({CORE_END - CORE_START} bp)')
print(f'Hamming distance: {sum(a != b for a, b in zip(full_wt_seq, full_mut_seq))}')

wildtype_seq = full_wt_seq
mutant_seq = full_mut_seq

In [ ]:
# Cell: Generate Single-Point Mutation Walk

def single_point_mutation_walk(seq_start, seq_end, seed=SEED):
    """Create a mutation walk from seq_start to seq_end, one base at a time."""
    assert len(seq_start) == len(seq_end), 'Sequences must be same length'

    diff_positions = [i for i in range(len(seq_start))
                      if seq_start[i] != seq_end[i]]

    rng = np.random.default_rng(seed)
    rng.shuffle(diff_positions)

    current = list(seq_start)
    walk = [''.join(current)]
    step_positions = []

    for pos in diff_positions:
        current[pos] = seq_end[pos]
        walk.append(''.join(current))
        step_positions.append(pos)

    assert walk[-1] == seq_end, 'Walk did not reach target'
    return walk, step_positions


mutation_walk, walk_positions = single_point_mutation_walk(wildtype_seq, mutant_seq)

n_steps = len(mutation_walk)
print(f'Mutation walk: {n_steps} steps (start + {n_steps - 1} mutations)')

center_pos = (CORE_START + CORE_END) // 2
pathogenic_step = None
for i, pos in enumerate(walk_positions):
    if pos == center_pos:
        pathogenic_step = i + 1
        break
print(f'Pathogenic C61G mutation occurs at step {pathogenic_step}/{n_steps - 1}')

walk_cache = f'{CACHE_DIR}/brca1_mutation_walk.npz'
np.savez(walk_cache, walk=mutation_walk, positions=walk_positions,
         pathogenic_step=pathogenic_step)
print(f'Walk cached to {walk_cache}')

In [ ]:
# Cell: Load Caduceus (Full Pattern with Weight Tying)
#
# Caduceus requires special handling:
#   - Patching transformers for compatibility
#   - Respecting native rcps flag (PS models are RC-equivariant)
#   - Disabling fused Triton norms if unavailable
#   - Re-tying mamba_rev <-> mamba_fwd weights after loading
#   - RC-invariant pooling for PS models

import torch
import gc
from transformers import AutoModel, AutoTokenizer, AutoConfig
import transformers.modeling_utils as _mu


def _patch_tied_weights():
    """Patch transformers for Caduceus compatibility."""
    if getattr(_mu, '_caduceus_patched', False):
        return
    orig_mark = _mu.PreTrainedModel.mark_tied_weights_as_initialized
    def safe_mark(self):
        if not hasattr(self, 'all_tied_weights_keys'):
            self.all_tied_weights_keys = {}
        return orig_mark(self)
    _mu.PreTrainedModel.mark_tied_weights_as_initialized = safe_mark

    orig_finalize = _mu.PreTrainedModel._finalize_load_state_dict
    @staticmethod
    def safe_finalize(model, load_config, load_info):
        if not hasattr(model, 'all_tied_weights_keys'):
            model.all_tied_weights_keys = {}
        orig_tie = model.tie_weights
        def _patched_tie(missing_keys=None, recompute_mapping=False, **kwargs):
            return orig_tie()
        model.tie_weights = _patched_tie
        return orig_finalize(model, load_config, load_info)
    _mu.PreTrainedModel._finalize_load_state_dict = safe_finalize
    _mu._caduceus_patched = True

_patch_tied_weights()
print('Patched transformers for Caduceus compatibility')


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def load_caduceus(model_name=CADUCEUS_MODEL, batch_size=4):
    """Load Caduceus model and return embedding function."""
    print(f'Loading {model_name}...')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
    is_rcps = getattr(cfg, 'rcps', False)
    print(f'  Config: rcps={is_rcps}, bidirectional={getattr(cfg, "bidirectional", "N/A")}')

    # Disable fused Triton norms if unavailable
    try:
        from mamba_ssm.ops.triton.layer_norm import layer_norm_fn, rms_norm_fn
        if layer_norm_fn is None or rms_norm_fn is None:
            raise ImportError
    except (ImportError, AttributeError):
        if hasattr(cfg, 'fused_add_norm'):
            cfg.fused_add_norm = False
        if hasattr(cfg, 'fused_dropout_add_ln'):
            cfg.fused_dropout_add_ln = False

    model = AutoModel.from_pretrained(
        model_name, config=cfg, trust_remote_code=True,
    ).to(device).eval()

    # Re-tie weights
    if hasattr(model, 'tie_weights'):
        model.tie_weights()

    # Verify + manual re-tie if needed
    try:
        layer0_mixer = model.backbone.layers[0].mixer
        inner = layer0_mixer.submodule if hasattr(layer0_mixer, 'submodule') else layer0_mixer
        fwd_ptr = inner.mamba_fwd.in_proj.weight.data_ptr()
        rev_ptr = inner.mamba_rev.in_proj.weight.data_ptr()
        tied = fwd_ptr == rev_ptr
        print(f'  Weight tie: {"OK" if tied else "BROKEN -- forcing re-tie"}')
        if is_rcps and not tied:
            for layer in model.backbone.layers:
                mix = layer.mixer.submodule if hasattr(layer.mixer, 'submodule') else layer.mixer
                mix.mamba_rev.in_proj.weight = mix.mamba_fwd.in_proj.weight
                if hasattr(mix.mamba_fwd.in_proj, 'bias') and mix.mamba_fwd.in_proj.bias is not None:
                    mix.mamba_rev.in_proj.bias = mix.mamba_fwd.in_proj.bias
                mix.mamba_rev.out_proj.weight = mix.mamba_fwd.out_proj.weight
                if hasattr(mix.mamba_fwd.out_proj, 'bias') and mix.mamba_fwd.out_proj.bias is not None:
                    mix.mamba_rev.out_proj.bias = mix.mamba_fwd.out_proj.bias
    except Exception as e:
        print(f'  Weight tie check skipped: {e}')

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'  Params: {n_params:.1f}M')

    @torch.no_grad()
    def embed(sequences):
        embeddings = []
        n_batches = (len(sequences) + batch_size - 1) // batch_size
        for i in range(0, len(sequences), batch_size):
            batch = sequences[i:i + batch_size]
            batch_idx = i // batch_size + 1
            if batch_idx % 10 == 0 or batch_idx == n_batches:
                print(f'  Batch {batch_idx}/{n_batches}', end='\r')
            tokens = tokenizer(
                batch, return_tensors='pt', padding=True,
                truncation=True, max_length=131072,
            )
            tokens = {k: v.to(device) for k, v in tokens.items()}
            outputs = model(**tokens, output_hidden_states=True)
            hidden = outputs.hidden_states[-1]
            # RC-invariant pooling for PS models
            if is_rcps:
                d = hidden.shape[-1]
                half1 = hidden[..., :d//2]
                half2 = hidden[..., d//2:]
                hidden = (half1 + half2.flip(dims=[-1])) / 2
            if 'attention_mask' in tokens:
                mask = tokens['attention_mask'].unsqueeze(-1)
                pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            else:
                pooled = hidden.mean(dim=1)
            embeddings.append(pooled.cpu().numpy())
        print()
        return np.concatenate(embeddings, axis=0)

    return embed, model, tokenizer, n_params


caduceus_embed, caduceus_model, caduceus_tokenizer, caduceus_params = load_caduceus()
print(f'Caduceus ready: {caduceus_params:.1f}M params')

In [ ]:
# Cell: Embed Mutation Walk with Caduceus

caduceus_cache = f'{CACHE_DIR}/caduceus_brca1_walk.npy'

if os.path.exists(caduceus_cache):
    print('Loading cached Caduceus embeddings...')
    caduceus_embeddings = np.load(caduceus_cache)
else:
    print(f'Embedding {len(mutation_walk)} walk steps with Caduceus...')
    caduceus_embeddings = caduceus_embed(mutation_walk)
    np.save(caduceus_cache, caduceus_embeddings)
    print(f'Cached to {caduceus_cache}')

print(f'Caduceus embeddings: {caduceus_embeddings.shape}')

# Free GPU memory
del caduceus_model
cleanup_gpu()
print('Caduceus model freed from GPU')

In [ ]:
# Cell: Quick Sanity Check

from scipy.spatial.distance import cosine

print('Sanity check: cosine distance from wildtype')
for step_idx in [0, 10, 50, len(caduceus_embeddings) - 1]:
    if step_idx < len(caduceus_embeddings):
        d = cosine(caduceus_embeddings[0], caduceus_embeddings[step_idx])
        print(f'  Step {step_idx:4d}: cosine dist = {d:.6f}')

# Consecutive step distances
dists = [cosine(caduceus_embeddings[i], caduceus_embeddings[i + 1])
         for i in range(len(caduceus_embeddings) - 1)]
dists = np.array(dists)
print(f'\nConsecutive step cosine distances:')
print(f'  mean={dists.mean():.6f}, std={dists.std():.6f}, max={dists.max():.6f}')
print(f'  smoothness (mean/max): {dists.mean() / (dists.max() + 1e-8):.4f}')
print(f'\nCaduceus embeddings cached and ready for analysis notebook.')